In [ ]:
import pandas as pd

#1: Load the dataset and select one commodity
# Load the cleaned dataset
cleaned_df = pd.read_csv(
    "../data/processed/cleaned_farmgate_prices.csv",
    parse_dates=["Date"]
)

# Select our first commodity
selected_commodity = "Palay [Paddy] Other Variety, dry (conv. to 14% mc)"

commodity_data = cleaned_df[
    (cleaned_df["Category"] == "Cereals") &
    (cleaned_df["Commodity"] == selected_commodity)
].sort_values("Date").copy()

# Check the selected commodity
print("Total months:", len(commodity_data))
print("First month:", commodity_data["Date"].min())
print("Last month:", commodity_data["Date"].max())
print("Interpolated months:", commodity_data["Was_Interpolated"].sum())

display(commodity_data.head())

In [ ]:
#2. Prepare the time series and split the data (80/20)
from sktime.split import temporal_train_test_split

# Prepare the monthly price series
monthly_prices = commodity_data.set_index("Date")["Price"].asfreq("MS")
# Convert dates into monthly periods for sktime
monthly_prices.index = monthly_prices.index.to_period("M")


# Chronological 80% training and 20% testing
train_data, test_data = temporal_train_test_split(
    monthly_prices, test_size=0.20
)

print("Training months:", len(train_data))
print("Testing months:", len(test_data))
print("Missing prices:", monthly_prices.isna().sum())

In [ ]:
test_data

In [ ]:
#3. Implement Seasonal Naïve
from sktime.forecasting.naive import NaiveForecaster

# Create the Seasonal Naïve model
seasonal_naive = NaiveForecaster(strategy="last", sp=12)

# Train using historical prices
seasonal_naive.fit(train_data)

# Forecast the next 6 months
seasonal_forecast = seasonal_naive.predict(fh=range(1, 7))

# Compare forecasts with actual test prices
comparison = pd.DataFrame({
    "Actual": test_data.iloc[:6],
    "Forecast": seasonal_forecast
})

display(comparison)

In [ ]:
# 4. Implement Holt-Winters
from sktime.forecasting.exp_smoothing import ExponentialSmoothing

# Create the Holt-Winters model
holt_winters = ExponentialSmoothing(
    trend="add",
    seasonal="add",
    sp=12
)

# Train using the same training data
holt_winters.fit(train_data)

# Forecast the next 6 months
holt_forecast = holt_winters.predict(fh=range(1, 7))

# Compare predictions with actual prices
holt_comparison = pd.DataFrame({
    "Actual": test_data.iloc[:6],
    "Forecast": holt_forecast
})

display(holt_comparison)

In [ ]:
# 5. Implement AutoARIMA

from sktime.forecasting.arima import AutoARIMA

# Create the AutoARIMA model
auto_arima = AutoARIMA(
    seasonal=False,
    stepwise=True,
    suppress_warnings=True
)

# Train using the same training data
auto_arima.fit(train_data)

# Forecast the next 6 months
arima_forecast = auto_arima.predict(fh=range(1, 7))

# Compare predictions with actual prices
arima_comparison = pd.DataFrame({
    "Actual": test_data.iloc[:6],
    "Forecast": arima_forecast
})

display(arima_comparison)

In [ ]:
#6. Compare the Three Models

# Combine the forecasts from all three models
model_comparison = pd.DataFrame({
    "Actual": test_data.iloc[:6],
    "Seasonal Naive": seasonal_forecast,
    "Holt-Winters": holt_forecast,
    "AutoARIMA": arima_forecast
})

# Round prices to 2 decimal places
display(model_comparison.round(2))

In [ ]:
#7. Evaluate Using MAE and RMSE

import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Evaluate the three forecasting models
evaluation_results = []

for model_name in ["Seasonal Naive", "Holt-Winters", "AutoARIMA"]:
    actual = model_comparison["Actual"]
    predicted = model_comparison[model_name]

    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))

    evaluation_results.append({
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse
    })

# Display model performance
evaluation_df = pd.DataFrame(evaluation_results)

display(evaluation_df.round(2))

In [ ]:
#9. Prepare Six-Month Backtesting Windows

# Divide testing data into complete 6-month windows
test_windows = []
for start in range(0, len(test_data) - 5, 6):
    window = test_data.iloc[start:start + 6]
    test_windows.append(window)


# Display each testing period
for number, window in enumerate(test_windows, 1):
    print(f"Window {number}: {window.index[0]} to {window.index[-1]}")

print("Total windows:", len(test_windows))

In [ ]:
# Reusable function for six-month backtesting
def backtest_model(model, monthly_prices, test_windows):
    backtest_results = []

    for test_window in test_windows:
        # Get all historical prices before the testing period
        training_history = monthly_prices[
            monthly_prices.index < test_window.index[0]
        ]

        # Create and train a fresh model
        current_model = model.clone()
        current_model.fit(training_history)

        # Forecast the next 6 months
        predicted_prices = current_model.predict(fh=range(1, 7))

        # Calculate forecasting errors
        backtest_results.append({
            "Test Period": f"{test_window.index[0]} to {test_window.index[-1]}",
            "MAE": mean_absolute_error(test_window, predicted_prices),
            "RMSE": np.sqrt(mean_squared_error(test_window, predicted_prices))
        })

    return pd.DataFrame(backtest_results)

In [19]:
# Backtest Seasonal Naïve
seasonal_results = backtest_model(
    NaiveForecaster(strategy="last", sp=12),
    monthly_prices,
    test_windows
)

display(seasonal_results.round(2))

,Test Period,MAE,RMSE
0,2023-06 to 2023-11,3.07,3.30
1,2023-12 to 2024-05,6.22,6.26
2,2024-06 to 2024-11,2.79,3.45
3,2024-12 to 2025-05,5.17,5.44
4,2025-06 to 2025-11,6.29,6.53
5,2025-12 to 2026-05,3.34,3.74


In [ ]:
# Backtest Holt-Winters using the same six testing windows
holt_results = backtest_model(
    ExponentialSmoothing(
        trend="add",
        seasonal="add",
        sp=12
    ),
    monthly_prices,
    test_windows
)

# Display Holt-Winters evaluation results
display(holt_results.round(2))


,Test Period,MAE,RMSE
0,2023-06 to 2023-11,1.80,2.29
1,2023-12 to 2024-05,1.80,1.90
2,2024-06 to 2024-11,2.00,2.46
3,2024-12 to 2025-05,1.89,2.43
4,2025-06 to 2025-11,1.34,1.84
5,2025-12 to 2026-05,3.86,4.24


In [ ]:

# Backtest AutoARIMA using the same six testing windows
arima_results = backtest_model(
    AutoARIMA(
        seasonal=False,
        stepwise=True,
        suppress_warnings=True
    ),
    monthly_prices,
    test_windows
)

# Display AutoARIMA evaluation results
display(arima_results.round(2))


In [22]:

# Store the backtesting results of all three models
all_model_results = {
    "Seasonal Naive": seasonal_results,
    "Holt-Winters": holt_results,
    "AutoARIMA": arima_results
}

# Calculate overall evaluation metrics
summary_results = []

for model_name, results in all_model_results.items():
    summary_results.append({
        "Model": model_name,
        "MAE": results["MAE"].mean(),
        "RMSE": np.sqrt((results["RMSE"] ** 2).mean())
    })

# Create a comparison table, ordered by lowest MAE
summary_df = pd.DataFrame(summary_results).sort_values("MAE")

display(summary_df.round(2))


,Model,MAE,RMSE
1,Holt-Winters,2.12,2.65
2,AutoARIMA,2.15,2.77
0,Seasonal Naive,4.48,4.97
